## Install Libraries

Translates the Vietnamese gold JSONL (`vi_sentence_str`, `vi_term`) into
English with a local Qwen checkpoint, and writes a new JSONL into the same
existing Drive folder.

The output keeps every original field. It only adds:

- `en_sentence_str` next to `vi_sentence_str`,
- `en_term` per entity, next to `vi_term`,
- `en_start_token_idx` / `en_end_token_idx`, recomputed by matching the
  translated term against the whitespace tokens of the translated sentence.
  When the model paraphrases the term instead of reusing it, the span is
  written as `null` and counted in the summary, so a silent half-translated
  dataset cannot pass as complete.

The notebook needs no repository clone: it reads and writes JSONL only.
Translations are cached in `CACHE_PATH` and appended as they arrive, so a
disconnected runtime resumes instead of starting over.


In [ ]:
%pip install "tqdm>=4.66" "transformers>=4.49" "accelerate>=0.34" "bitsandbytes>=0.43"


## Import Libraries

`torch` and `transformers` are imported here; the model itself is loaded in
the model stage so a configuration mistake fails before the weights download.


In [ ]:
from __future__ import annotations

import gc
import json
import os
import re
import sys
import time
from pathlib import Path

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer

print("Python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## Constants and Configuration

Set `DRIVE_DATA_DIR` to the folder that already holds the gold JSONL; the
notebook refuses to create it, so a typo cannot scatter files over Drive.

`OUTPUT_PATH` is the translated dataset, `CACHE_PATH` is the append-only
translation memory used to resume. Both live in that same folder.

`LIMIT = None` translates every record; set an int for a smoke run.


In [ ]:
# ---------------------------------------------------------------------------
# Google Drive input and output (REQUIRED: set DRIVE_DATA_DIR below)
# ---------------------------------------------------------------------------
MOUNT_DRIVE = True
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
DRIVE_DATA_DIR = DRIVE_ROOT / "REPLACE_ME"                      # TODO: set once
SOURCE_DATA_PATH = DRIVE_DATA_DIR / "test_fixed_verified_gold_revised.jsonl"
OUTPUT_DIR = DRIVE_DATA_DIR                                     # must already exist
OUTPUT_PATH = OUTPUT_DIR / "test_fixed_verified_gold_revised_en.jsonl"
CACHE_PATH = OUTPUT_DIR / "vi_en_translation_cache.jsonl"

# ---------------------------------------------------------------------------
# Translation
# ---------------------------------------------------------------------------
SOURCE_LANGUAGE = "Vietnamese"
TARGET_LANGUAGE = "English"
BATCH_SIZE = 8                  # strings per generate() call
MAX_INPUT_TOKENS = 512          # prompt cap; a truncated prompt is a bug
MAX_NEW_TOKENS = 256            # one sentence or term, never long
LIMIT = None                    # None = all records; an int truncates

# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
MODEL_ID = "Qwen/Qwen3.8-27B"   # Hub id or a local directory
LOAD_IN_4BIT = True             # 27B in bf16 is ~54 GB; 4-bit is ~15 GB
DEVICE_MAP = "auto"
TRUST_REMOTE_CODE = False

# ---------------------------------------------------------------------------
# Runtime
# ---------------------------------------------------------------------------
CACHE_DIR = Path("/content/grf_cache")
HF_HOME = CACHE_DIR / "huggingface"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_HOME))

_START = time.time()


def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - _START:6.1f}s] {message}", flush=True)


def require_configured(value, name: str):
    text = str(value)
    if "REPLACE_ME" in text:
        raise ValueError(f"{name} is still a placeholder: {text!r}. Set it in the constants cell.")
    return value


log("Configuration loaded.")
print(f"Input data   : {SOURCE_DATA_PATH}")
print(f"Output       : {OUTPUT_PATH}")
print(f"Cache        : {CACHE_PATH}")
print(f"Direction    : {SOURCE_LANGUAGE} -> {TARGET_LANGUAGE}")
print(f"Model        : {MODEL_ID} (4-bit={LOAD_IN_4BIT})")


## Stage 1 — Mount Drive and Locate the Existing Folder

Both the input and the two output files live in the folder you configured;
nothing is created outside it. The folder must exist already.


In [ ]:
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:  # running outside Colab
        raise RuntimeError(
            "Google Colab is required to mount Drive. Run this notebook in Colab, "
            "or set MOUNT_DRIVE = False only if the inputs are already local to the runtime."
        ) from exc
    drive.mount(str(DRIVE_MOUNT_POINT))

source_path = Path(require_configured(SOURCE_DATA_PATH, "SOURCE_DATA_PATH"))
output_dir = Path(require_configured(OUTPUT_DIR, "OUTPUT_DIR"))
output_path = Path(OUTPUT_PATH)
cache_path = Path(CACHE_PATH)

if not source_path.is_file():
    raise FileNotFoundError(f"Gold data not found: {source_path}")
if not output_dir.is_dir():
    raise FileNotFoundError(
        f"OUTPUT_DIR does not exist: {output_dir}. Point it at the existing folder; "
        "this notebook never creates it."
    )
for label, path in (("OUTPUT_PATH", output_path), ("CACHE_PATH", cache_path)):
    if path.parent != output_dir:
        raise ValueError(f"{label} must live in OUTPUT_DIR, got {path.parent}")
log(f"Input: {source_path} ({source_path.stat().st_size / 1024 ** 2:.2f} MB)")
log(f"Output dir: {output_dir}")


## Stage 2 — Load the Gold Records

Every non-blank line must be an object with `entities` and
`vi_sentence_str`; a malformed record fails here instead of producing a
half-translated file.


In [ ]:
log(f"Loading gold records from {source_path} ...")
records = []
with source_path.open(encoding="utf-8-sig") as handle:
    for line_number, line in enumerate(handle, 1):
        if not line.strip():
            continue
        where = f"{source_path}:{line_number}"
        try:
            record = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(f"{where}: invalid JSON: {exc.msg}") from exc
        if not isinstance(record, dict):
            raise ValueError(f"{where}: record must be an object")
        sentence = record.get("vi_sentence_str")
        entities = record.get("entities")
        if not isinstance(sentence, str) or not sentence.strip():
            raise ValueError(f"{where}: vi_sentence_str must be a non-empty string")
        if not isinstance(entities, list) or not entities:
            raise ValueError(f"{where}: entities must be a non-empty list")
        for entity_index, entity in enumerate(entities):
            term = entity.get("vi_term") if isinstance(entity, dict) else None
            if not isinstance(term, str) or not term.strip():
                raise ValueError(f"{where}:entities[{entity_index}]: vi_term must be a non-empty string")
        records.append(record)
        if LIMIT is not None and len(records) >= LIMIT:
            break

if not records:
    raise ValueError(f"{source_path} produced no records.")
n_entities = sum(len(record["entities"]) for record in records)
log(f"Loaded {len(records):,} records with {n_entities:,} entities.")


## Stage 3 — Collect the Unique Strings

Terms repeat heavily across records, so the work list is deduplicated: one
translation per distinct string, then the dataset is assembled from that
dictionary. This is also what makes the cache useful across reruns.


In [ ]:
texts = []
seen = set()
for record in records:
    candidates = [record["vi_sentence_str"]]
    candidates.extend(entity["vi_term"] for entity in record["entities"])
    for text in candidates:
        key = " ".join(text.split())
        if key and key not in seen:
            seen.add(key)
            texts.append(key)
log(f"Unique strings to translate: {len(texts):,} (from {n_entities + len(records):,} slots).")

cache = {}
if cache_path.is_file():
    with cache_path.open(encoding="utf-8-sig") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                item = json.loads(line)
                cache[item["source"]] = item["target"]
            except (json.JSONDecodeError, KeyError, TypeError) as exc:
                raise ValueError(f"{cache_path}:{line_number}: bad cache record: {exc}") from exc
    log(f"Resumed from cache: {len(cache):,} translations already on disk.")
else:
    log("No cache yet; every string will be translated.")

pending = [text for text in texts if not cache.get(text)]
log(f"Pending: {len(pending):,} strings.")


## Stage 4 — Load the Model

**Memory.** 27B parameters is ~54 GB in bf16 and ~15 GB in 4-bit, plus the KV
cache, so a 15 GB Colab GPU cannot hold the 4-bit build reliably. Use an
A100/L40S-class GPU, or set `MODEL_ID` to a smaller Qwen checkpoint. The
notebook logs the real footprint after loading instead of guessing.

The checkpoint is multimodal, so `AutoModelForImageTextToText` is tried first
and `AutoModelForCausalLM` is the fallback for a text-only export. Generation
itself is text-only: no images are passed.


In [ ]:
if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    log(f"GPU: {torch.cuda.get_device_name(0)} "
        f"({total_b / 1024 ** 3:.1f} GiB total, {free_b / 1024 ** 3:.1f} GiB free)")
    if not LOAD_IN_4BIT and total_b / 1024 ** 3 < 60:
        log("WARNING: 27B in bf16 needs ~54 GB of weights; this GPU is too small. "
            "Set LOAD_IN_4BIT = True or pick a smaller checkpoint.")
else:
    log("WARNING: no CUDA device; a 27B model on CPU is not practical.")

log(f"Loading tokenizer and model {MODEL_ID!r} (first run downloads weights) ...")
started = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=TRUST_REMOTE_CODE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # required for batched decoder-only generation

load_kwargs = {
    "trust_remote_code": TRUST_REMOTE_CODE,
    "device_map": DEVICE_MAP,
    "torch_dtype": "auto",
}
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

try:
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **load_kwargs)
except (ImportError, ValueError) as exc:
    log(f"AutoModelForImageTextToText unavailable ({exc.__class__.__name__}); "
        "falling back to AutoModelForCausalLM.")
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()

log(
    f"Model ready in {time.time() - started:.1f}s; "
    f"weights footprint {model.get_memory_footprint() / 1024 ** 3:.2f} GiB."
)


## Stage 5 — Translate the Pending Strings

Strings are batched, generated greedily (`do_sample=False`) and cleaned:
any `thinking` block is removed and the answer is collapsed to one line.
Each finished translation is appended to the cache and flushed immediately,
so killing the runtime costs at most one batch.

An empty answer is not cached, so the next run retries it; it is reported at
the end instead of being written as a silent empty string.


In [ ]:
SYSTEM_PROMPT = (
    f"You are a professional biomedical translator. Translate the user's "
    f"{SOURCE_LANGUAGE} text into {TARGET_LANGUAGE}. Reply with the translation "
    "only: no notes, no quotes, no explanations. Keep gene symbols, drug names, "
    "abbreviations and numbers unchanged."
)
_THINK_BLOCK = re.compile(r"<think\b[^>]*>.*? response", re.DOTALL | re.IGNORECASE)


def build_prompt(text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def clean_translation(raw: str) -> str:
    text = _THINK_BLOCK.sub("", raw).strip()
    text = text.strip().strip('"').strip("'").strip()
    return " ".join(text.split())


def translate_batch(batch):
    prompts = [build_prompt(text) for text in batch]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = output[:, inputs["input_ids"].shape[1]:]
    return [
        clean_translation(text)
        for text in tokenizer.batch_decode(generated, skip_special_tokens=True)
    ]


failures = []
started = time.time()
if pending:
    handle = cache_path.open("a", encoding="utf-8")
    try:
        with tqdm(total=len(pending), desc="Translating", unit="string") as bar:
            for start in range(0, len(pending), BATCH_SIZE):
                batch = pending[start:start + BATCH_SIZE]
                try:
                    results = translate_batch(batch)
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    log(f"OOM on a batch of {len(batch)}; retrying one string at a time.")
                    results = []
                    for text in batch:
                        try:
                            results.extend(translate_batch([text]))
                        except torch.cuda.OutOfMemoryError:
                            torch.cuda.empty_cache()
                            results.append("")
                for text, result in zip(batch, results):
                    if not result:
                        failures.append(text)
                        continue
                    cache[text] = result
                    handle.write(json.dumps({"source": text, "target": result}, ensure_ascii=False) + "\n")
                    handle.flush()
                bar.update(len(batch))
    finally:
        handle.close()

elapsed = time.time() - started
log(
    f"Translated {len(pending) - len(failures):,} of {len(pending):,} pending strings "
    f"in {elapsed:.1f}s ({len(pending) / max(elapsed, 1e-6):.2f} strings/s)."
)
if failures:
    log(f"WARNING: {len(failures):,} empty answer(s); first: {failures[0]!r}. "
        "They are not cached and will be retried on the next run.")


## Stage 6 — Assemble the Translated Dataset

Every original field is copied unchanged. `en_term` and `en_sentence_str` are
added, and the English token span is recomputed by matching the translated
term against the whitespace tokens of the translated sentence. A paraphrase
that breaks the match leaves the span `null` and increments the counter.


In [ ]:
def locate_span(tokens, term):
    """Return the (start, end) token span of term inside tokens, or (None, None)."""
    wanted = term.lower().split()
    if not wanted:
        return None, None
    width = len(wanted)
    for start in range(len(tokens) - width + 1):
        if tokens[start:start + width] == wanted:
            return start, start + width - 1
    return None, None


translated_records = []
unmatched = []
missing = 0
for record_index, record in enumerate(records):
    sentence = cache.get(" ".join(record["vi_sentence_str"].split()))
    if not sentence:
        missing += 1
        continue
    tokens = sentence.lower().split()
    entities = []
    for entity_index, entity in enumerate(record["entities"]):
        term = cache.get(" ".join(entity["vi_term"].split()))
        if not term:
            missing += 1
            continue
        start, end = locate_span(tokens, term)
        if start is None:
            unmatched.append((record_index, entity_index, term))
        entities.append({
            **entity,
            "en_term": term,
            "en_start_token_idx": start,
            "en_end_token_idx": end,
        })
    translated_records.append({
        **record,
        "en_sentence_str": sentence,
        "entities": entities,
    })

if missing:
    raise RuntimeError(
        f"{missing:,} string(s) still have no translation. Re-run stage 5 until "
        "it reports zero pending strings, then assemble again."
    )

with output_path.open("w", encoding="utf-8") as handle:
    for record in translated_records:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

log(
    f"Wrote {len(translated_records):,} records "
    f"({output_path.stat().st_size / 1024 ** 2:.2f} MB) to {output_path}."
)
if unmatched:
    log(
        f"WARNING: {len(unmatched):,} entity span(s) could not be located in the "
        f"translated sentence and are null; first: record {unmatched[0][0]} "
        f"entity {unmatched[0][1]} term {unmatched[0][2]!r}."
    )
else:
    log("All entity spans were located in the translated sentences.")


## Stage 7 — Verify and Release Memory

Reload the written file and check that every record carries both sentences and
that the counts match the input, then drop the model, the tokenizer and the
records and empty the CUDA cache.


In [ ]:
log(f"Reloading {output_path} ...")
with output_path.open(encoding="utf-8") as handle:
    reloaded = [json.loads(line) for line in handle if line.strip()]
assert len(reloaded) == len(translated_records), (len(reloaded), len(translated_records))
assert all(record.get("en_sentence_str") for record in reloaded)
assert all(entity.get("en_term") for record in reloaded for entity in record["entities"])
log(f"Verified: {len(reloaded):,} translated records, cache {len(cache):,} strings.")

for name in ("model", "tokenizer", "records", "translated_records", "reloaded", "inputs", "output"):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

log("Translation complete; memory released.")
